<a href="https://colab.research.google.com/github/bmf87/pydata_copilot/blob/main/notebooks/dpo_train_qwen2_5_coder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS 552 - Generative AI
* Capstone Project Title: PyData Copilot
* Objective: Direct Preference Optimization (DPO) of Qwen2.5-Coder-7B-Instruct
* Author: Brett Favro


Notebook uses [DPOTrainer](https://huggingface.co/docs/trl/en/dpo_trainer) on [Qwen/Qwen2.5-Coder-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct) to align model to a custom Python coding preference dataset focuses on data analysis and EDA.

* Model Parameters: 7.61B
* Model Layers: 28
* Attention Heads: 28 for Q, 4 for KV
* Context Length: 131K

***Notes***:
Qwen models do not train on PAD tokens, while DPOTrainer requires them.
We map the Tokenizers EOS to PAD and BOS, then sync this to the model's config.

## Notebook Dependencies

In [ ]:
!pip install -q -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 39.9 MB/s eta 0:00:00


In [ ]:
import torch
import multiprocessing
from dataclasses import dataclass
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
from google.colab import drive, userdata
from huggingface_hub import login

In [ ]:
# Mount Google Drive
drive.mount('/content/drive/')
project_dir = '/content/drive/MyDrive/Colab Notebooks/DS552/project'
data_dir = project_dir + '/data'
model_dir = project_dir + '/model'

Mounted at /content/drive/


## TrainerConfig: Dataclass for Training Configuration
* Encapsulates all training hyperparameters

In [ ]:
@dataclass
class TrainerConfig:
    """Configuration for the DPO Pipeline"""
    #model_name: str = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    model_name: str = "Qwen/Qwen2.5-Coder-7B-Instruct"
    dataset_path: str = data_dir + "/pandas_dpo_dataset_final.jsonl"
    output_dir: str = model_dir
    final_merged_dir: str = model_dir + "/dpo-merged"

    # LoRA Config
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # Training Config
    epochs: int = 3               # 1 - 3: DPO standard
    batch_size: int = 2
    grad_accum_steps: int = 8     # calc gradients every n steps
    beta: float = 0.1             # KL Penalty/Regularization
    lr: float = 5e-6              # Learning Rate: 5 x 10^{-6} or 0.000005: low/safe for stable convergence


## Console Logger
* Setup callback to enable real-time metric display

In [ ]:
from transformers import TrainerCallback

class ConsoleLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            print(f"[LOG step={state.global_step} epoch={state.epoch}] {logs}")

## QwenDPOTrainerPipeline: Encapsulate Training Pipeline
* Pipeline encapsulates end-to-end workflow, covering:
  * Tokenization
  * Dataset preparation
  * Model setup
  * Model training
  * Model merging
* Auto Device Mapping - enables running on GPU or CPU

In [ ]:
class QwenDPOTrainerPipeline:
    def __init__(self, config: TrainerConfig):
        self.config = config
        self.tokenizer = None
        self.model = None
        self.train_dataset = None
        self.eval_dataset = None
        self.trainer = None

    def _format_chat_template(self, example):
        """
        Formats the given example into Qwen's ChatML template for DPO training.

        This method takes an example dictionary containing 'prompt', 'chosen', and 'rejected'
        fields, and formats them into the ChatML format expected by the tokenizer.
        It prepares the prompt, chosen response, and rejected response
        texts by applying the tokenizer's chat template, extracting the assistant's
        parts for chosen and rejected responses.

        Args:
            example (dict): A dictionary with 'prompt', 'chosen', and 'rejected' keys.

        Returns:
            dict: A dictionary containing the formatted 'prompt', 'chosen', and 'rejected'
                  texts suitable for DPO training.
        """
        prompt_msgs = [{"role": "user", "content": example["prompt"]}]
        chosen_msgs = [{"role": "assistant", "content": example["chosen"]}]
        rejected_msgs = [{"role": "assistant", "content": example["rejected"]}]

        prompt_text = self.tokenizer.apply_chat_template(
            prompt_msgs, tokenize=False, add_generation_prompt=True
        )

        chosen_text = self.tokenizer.apply_chat_template(
            prompt_msgs + chosen_msgs, tokenize=False
        )[len(prompt_text):]

        rejected_text = self.tokenizer.apply_chat_template(
            prompt_msgs + rejected_msgs, tokenize=False
        )[len(prompt_text):]

        return {
            "prompt": prompt_text,
            "chosen": chosen_text,
            "rejected": rejected_text
        }

    def setup_tokenizer(self):
        print("Setting up Tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
          self.tokenizer.pad_token = self.tokenizer.eos_token
        if self.tokenizer.bos_token is None:
          self.tokenizer.bos_token = self.tokenizer.eos_token

    def prepare_dataset(self, test_size=0.1):
        print("Preparing DPO dataset...")
        dataset = load_dataset("json", data_files=self.config.dataset_path, split="train")
        dataset = dataset.map(self._format_chat_template, num_proc=multiprocessing.cpu_count())
        dataset = dataset.train_test_split(test_size=test_size, seed=42)

        self.train_dataset = dataset["train"]
        self.eval_dataset = dataset["test"]
        print(f"Training dataset size: {len(self.train_dataset)}")
        print(f"Eval dataset size: {len(self.eval_dataset)}\n")

    def setup_model(self):
        print(f"Loading base model and applying LoraConfig...")

        HF_TOKEN = userdata.get('HF_TOKEN') # get token from Colab secrets
        if HF_TOKEN:
          login(token=HF_TOKEN)
          print("Successfully logged into Hugging Face!")

          base_model = AutoModelForCausalLM.from_pretrained(
            self.config.model_name,
            dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
          )
          print(f"Pulled base model:{self.config.model_name} from Hugging Face!\n")

          # DPOTrainer reads model.config during init & compares to tokenizer; mismatches trigger warnings
          # Qwen tokenizers often ship without pad or bos_token set (intentionally: model never trained on pad).
          # DPO needs them defined, so we've set pad_token = eos_token (ID 151643 for Qwen2.x) in setup_tokenizer
          base_model.config.pad_token_id = self.tokenizer.pad_token_id
          base_model.config.bos_token_id = self.tokenizer.bos_token_id
          base_model.config.eos_token_id = self.tokenizer.eos_token_id
          base_model.generation_config.pad_token_id = self.tokenizer.pad_token_id
          base_model.generation_config.bos_token_id = self.tokenizer.bos_token_id
          base_model.generation_config.eos_token_id = self.tokenizer.eos_token_id


          # Attach LoRA adapters to all linear layers
          # Attention Mechanism: Q, K, V, O (output)
          # MLP Layers: gate, up, down
          peft_config = LoraConfig(
            r=self.config.lora_r,
            lora_alpha=self.config.lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=self.config.lora_dropout,
            bias="none",
            task_type=TaskType.CAUSAL_LM
          )
          self.model = get_peft_model(base_model, peft_config)
        else:
          print("[ERROR] - Unable to setup model!")

    def train(self):
        print("Starting DPO Alignment (finetuning) phase...")
        training_args = DPOConfig(
            output_dir=self.config.output_dir,
            do_train=True,
            do_eval=True,
            eval_strategy="steps",
            eval_steps=500,
            save_steps=1000,
            logging_steps=50,
            logging_first_step=True,
            logging_strategy="steps",
            learning_rate=self.config.lr,
            per_device_train_batch_size=self.config.batch_size,
            gradient_accumulation_steps=self.config.grad_accum_steps,
            num_train_epochs=self.config.epochs,
            beta=self.config.beta,
            bf16=True,
            dataset_num_proc=multiprocessing.cpu_count(),
            disable_tqdm=False,
        )

        # ensure console logging
        #training_args.report_to = None
        print(training_args.report_to)

        self.trainer = DPOTrainer(
            model=self.model,
            ref_model=None,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            processing_class=self.tokenizer,
            callbacks=[ConsoleLoggingCallback()],
        )
        self.trainer.train()

    def evaluate_model(self):
        print("Running final evaluation on model...")
        self.trainer.evaluate()

    def merge_and_save(self):
        print("Training complete. Merging LoRA adapter into base model...")

        # self.model is a PeftModelForCausalLM (get_peft_model(base_model, peft_config))
        # Merge LoRA into the base model weights and drop adapter modules
        merged_model = self.model.merge_and_unload()
        # Save merged model + tokenizer
        merged_model.save_pretrained(self.config.final_merged_dir)
        self.tokenizer.save_pretrained(self.config.final_merged_dir)
        print(f"Merged model saved to {self.config.final_merged_dir}. Ready for Llama.cpp quantization!")

    def run_pipeline(self, test_size=0.1, final_eval=False):
        """
        Orchestrates end-to-end process.
        """
        self.setup_tokenizer()
        self.prepare_dataset(test_size)
        self.setup_model()
        self.train()
        if final_eval:
          self.evaluate_model()
        self.merge_and_save()



## **Train Model**: Run Pipeline
* **Objective**: model learns to distinguish between the chosen and rejected choices in the dataset based on the implicit DPO reward.

* Metrics Analyzed:
  * rewards/accuracies - proportion of training examples in a batch (mean over batch), where the chosen reward > corresponding rejected implicit reward (high 0.9x is common)
  * $\text{rewards/margins } (r_{\text{chosen}} - r_{\text{rejected}})$ - the average difference between the chosen and rejected implicit reward. This should increase relative to start of training.

* Model showed strong alignment:
  * **rewards/accuracies**: started at 0.81, then stabilized, finished training at: 0.957. Final eval: 0.942
  * **rewards/margins**: started at 0.154, then stabilized, finished training at: 4.208. Final eval: 4.090

In [ ]:
if __name__ == "__main__":
    # Can override params: (epochs=3, lora_r=32)
    config = TrainerConfig()
    pipeline = QwenDPOTrainerPipeline(config)
    print(f"CUDA Available: {torch.cuda.is_available()}")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    if torch.cuda.is_available():
      print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    # DPO datasets are $$$ - typical test sizes are 0.05 - 0.1
    pipeline.run_pipeline(test_size=0.1, final_eval=True)

CUDA Available: True
Number of GPUs available: 1
GPU Name: NVIDIA A100-SXM4-80GB
Setting up Tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Step,Training Loss,Validation Loss


[LOG step=1 epoch=0.006168080185042405] {'loss': 0.6931471824645996, 'grad_norm': 1.7165522575378418, 'learning_rate': 5e-06, 'entropy': 0.5098370015621185, 'num_tokens': 6526.0, 'logits/chosen': -1.3221185737061516, 'logits/rejected': -1.2879882935756712, 'mean_token_accuracy': 0.8036853522062302, 'rewards/chosen': 0.0, 'rewards/rejected': 0.0, 'rewards/accuracies': 0.0, 'rewards/margins': 0.0, 'logps/chosen': -141.69322967529297, 'logps/rejected': -129.04541778564453, 'epoch': 0.006168080185042405}
[LOG step=50 epoch=0.3084040092521203] {'loss': 0.6696813933703364, 'grad_norm': 1.643432855606079, 'learning_rate': 4.498977505112475e-06, 'entropy': 0.42559380023455134, 'num_tokens': 412651.0, 'logits/chosen': -1.3426133040531798, 'logits/rejected': -1.3924315460008216, 'mean_token_accuracy': 0.8096221121294158, 'rewards/chosen': 0.02757931222197153, 'rewards/rejected': -0.02244024506038915, 'rewards/accuracies': 0.7206632653061225, 'rewards/margins': 0.05001955711293481, 'logps/chosen'

[LOG step=489 epoch=3.0] {'eval_loss': 0.16978693008422852, 'eval_runtime': 77.2353, 'eval_samples_per_second': 3.742, 'eval_steps_per_second': 0.479, 'eval_entropy': 0.4287928632787756, 'eval_num_tokens': 3992580.0, 'eval_logits/chosen': -1.397194595681885, 'eval_logits/rejected': -1.4759027435267806, 'eval_mean_token_accuracy': 0.819090140832437, 'eval_rewards/chosen': 1.7930044998993744, 'eval_rewards/rejected': -1.5691501039105493, 'eval_rewards/accuracies': 0.9391891891891891, 'eval_rewards/margins': 3.362154660998164, 'eval_logps/chosen': -137.73589386811128, 'eval_logps/rejected': -150.0967568062447, 'epoch': 3.0}
Training complete. Merging LoRA adapter into base model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to /content/drive/MyDrive/Colab Notebooks/DS552/project/model/dpo-merged. Ready for Llama.cpp quantization!


## Push DPO Model to HuggingFace Repo

In [ ]:
#from huggingface_hub import whoami
#print(whoami())

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
hf_username = "bfavro73"
#dpo_model_name = "qwen2.5-coder-1.5b-pandas-dpo-aligned"
dpo_model_name = "qwen2.5-coder-7b-pandas-dpo-aligned"
HF_TOKEN = userdata.get('HF_TOKEN') # get token from Colab secrets
if HF_TOKEN:
  login(HF_TOKEN)
  print("Successfully logged into Hugging Face!")
  config = TrainerConfig()
  final_model = AutoModelForCausalLM.from_pretrained(config.final_merged_dir, trust_remote_code=True)
  final_tokenizer = AutoTokenizer.from_pretrained(config.final_merged_dir, trust_remote_code=True)

  final_model.push_to_hub(f"{hf_username}/{dpo_model_name}")
  final_tokenizer.push_to_hub(f"{hf_username}/{dpo_model_name}")
  print("Model and Tokenizer pushed to Hugging Face Hub")

Successfully logged into Hugging Face!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mply9cnrop/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Model and Tokenizer pushed to Hugging Face Hub


## Pull Model from HF Repo

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("bfavro73/qwen2.5-coder-1.5b-pandas-dpo-aligned")
tokenizer = AutoTokenizer.from_pretrained("bfavro73/qwen2.5-coder-1.5b-pandas-dpo-aligned")